In [63]:
import requests
from bs4 import BeautifulSoup

def scrape_wikipedia_biography(url):
    # Send a GET request to the Wikipedia page
    response = requests.get(url)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse the page content using BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')

        # Extract the title of the page
        title = soup.find('h1', id='firstHeading').text

        # Extract the first paragraph or infobox (optional)
        paragraphs = soup.find_all('p')
        bio = ""
        for paragraph in paragraphs:
            if paragraph.text.strip():
                bio += paragraph.text.strip() + "\n"
        
        # Return the title and biography content
        return title, bio
    else:
        print(f"Failed to retrieve page. Status code: {response.status_code}")
        return None, None


In [ ]:
from openai import OpenAI
import os
import json


# Create an instance of the OpenAI class
openai = OpenAI()

# generate the cp form data using the openai instance
# generate a prompt string containing all the schemas and ask it for the data of 100 persons
prompt_str = r"""
const cpMapEntrySchema = new mongoose.Schema({
  name: {
    type: String,
    required: true,
  },
  city: {
    type: String,
    required: true,
  },
  state: {
    type: String,
    required: true,
  },
  country: {
    type: String,
    required: true,
  },
  address: {
    type: String,
    required: true,
  },
  email: {
    type: String,
    required: true,
  },
  mobileNumber: {
    type: String,
    required: true,
    match: /^\+\d{1,4}\s\d{5}\s\d{5}$/,
  },
  personalWebsite: {
    type: String,
    required: false,
  },
  socialNetworkLink: [
    {
      type: String, 
      required: false,
    },
  ],
  gender: {
    type: String,
    required: true,
  },
  religion: {
    type: String,
    required: true,
  },
  caste: {
    type: String,
    required: true,
  },
  economicStatus: {
    type: String,
    required: true,
  },
  background: {
    type: String,
    required: true,
  },
  currentOccupation: {
    type: String,
    required: true,
  },
  educations: [
    {
      type: mongoose.Schema.Types.ObjectId,
      ref: "Education",
      required: true,
    },
  ],
  stages: [
    {
      type: mongoose.Schema.Types.ObjectId,
      ref: "Stage",
      required: true,
    },
  ],
  status: {
    type: String,
    required: true,
    default: "Moderation pending",
  },
  verified_by: {
    type: mongoose.Schema.Types.ObjectId,
    ref: "User",
    default: null,
    required: false,
  },
});

const skillSchema = new mongoose.Schema({
  skill: {
    type: String,
    required: true,
    unique: true,
  }
});

const skillEffortSchema = new mongoose.Schema({
    skill_id: {
        type: mongoose.Schema.Types.ObjectId,
        ref: "Skill",
        required: true,
    },
    effort: {
        type: String,
        required: true,
    },
    holder_id: {
        type: mongoose.Schema.Types.ObjectId,
        ref: "CPMapEntry",
        required: true,
    },
});

const stageSchema = new mongoose.Schema({
  stageName: {
    type: String,
    required: true,
  },
  stageType: {
    type: Number,
    required: true,
  },
  skill_ids: [
    {
      type: mongoose.Schema.Types.ObjectId,
      ref: "Skill",
      required: false,
    },
  ],
  learnings: {
    type: String,
    required: true,
  },
  from: {
    type: String,
    required: true,
    match: /^[0-9]{4}-(0[1-9]|1[0-2])$/,
  },
  to: {
    type: String,
    required: true,
    match: /^[0-9]{4}-(0[1-9]|1[0-2])$/,
  },
  motivation: {
    type: String,
  },
  suggestions: {
    type: String,
    required: true,
  },
  message: {
    type: String,
  },
});

const educationSchema = new mongoose.Schema({
  qualification_id: {
    type: mongoose.Schema.Types.ObjectId,
    ref: "Qualification",
    required: true,
  },
  qualificationName: {
	type: String,
	required: true
  },
  collegeName: {
    type: String,
    required: true,
  },
  collegeCity: {
    type: String,
    required: true,
  },
  from: {
    type: String,
    required: true,
    match: /^[0-9]{4}-(0[1-9]|1[0-2])$/,
  },
  to: {
    type: String,
    required: true,
    match: /^[0-9]{4}-(0[1-9]|1[0-2])$/,
  },
  collegeAddress: {
    type: String,
    required: true,
  },
});

Note these clearly: 
convert this biography to db schema data, based on the biography, fill the schema with the data and return the json data such that it can be used to fill the db schema
and dont hallucinate columns if they are not there, just make them null simply
give stage names more clearly and descriptive and education details in a single json object that has all the details
give the skills and effort required for the stage
give the entire data in the json format do not give any other statements, or any other punctuations like ``` or anything else
donot give this also Here's the JSON formatted schema derived from the biography for the database:

```
"""

data = []
People = {
    # just 5 enough in each category and preferably indian
    # "Business": ["Mukesh_Ambani", "Ratan_Tata", "Azim_Premji", "Lakshmi_Mittal", "Shiv_Nadar"],
    # "Film Industry": ["Amitabh_Bachchan", "Shah_Rukh_Khan", "Salman_Khan", "Aamir_Khan", "Rajinikanth"],
    # "Politicians": ["Narendra_Modi", "Indira_Gandhi", "Jawaharlal_Nehru", "Sardar_Vallabhbhai_Patel", "Atal_Bihari_Vajpayee"],
    # "Sports": ["Sachin_Tendulkar", "Virat_Kohli", "Milkha_Singh", "Pullela_Gopichand", "P._V._Sindhu"],
    # "Scientists": ["A._P._J._Abdul_Kalam", "C._V._Raman", "Vikram_Sarabhai", "Homi_J._Bhabha", "Satyendra_Nath_Bose"],
    # "Authors": ["Rabindranath_Tagore", "Jawaharlal_Nehru", "R._K._Narayan"],
    # "Musicians": ["A._R._Rahman", "Lata_Mangeshkar", "Kishore_Kumar", "Mohammed_Rafi", "M._S._Subbulakshmi"],
    # "Famous_Women": ["Indira_Gandhi", "Sarojini_Naidu", "Kalpana_Chawla", "Savitribai_Phule", "Rani_Lakshmibai"],
    # "Agriculture": ["Gurdev_Khush", "R._H._Richharia"], # "M._S._Swaminathan", "Norman_Borlaug",
}

for key in People:
	for person_title in People[key]:
		title, biography = scrape_wikipedia_biography('https://en.wikipedia.org/wiki/'+person_title)
		print(f"Title: {person_title}")

		response = openai.chat.completions.create(
			model="gpt-4",
			messages=[{"role": "system", "content": prompt_str + biography[:20000]}],
		)
		data.append(response.choices[0].message.content)

In [62]:
import json
from pymongo import MongoClient

# Establishing MongoDB connection
client = MongoClient('localhost', 27017)
db = client['cp']

for person in data:
    person = json.loads(person)
    # Adding default fields if not present
    with open('bio_all_data.json', 'a') as f:
        json.dump(person, f, indent=4)
    f.close()
    person['status'] = 'pending'
    person['verified_by'] = None

    # Processing education data
    edu = person['educations']
    person['educations'] = []  # Clear out the list for object IDs
    for ed in edu:
        # Insert each education entry into the 'education' collection
        ed_id = db['education'].insert_one(ed).inserted_id
        person['educations'].append(ed_id)  # Store the generated ObjectId
    
    # Processing career stages data, including skills and skillEffort
    stages = person['stages']
    person['stages'] = []  # Clear out the list for object IDs
    for stage in stages:
        # Handle skill_ids in each stage
        # skill_ids = stage.get('skill_ids', [])
        # stage['skill_ids'] = []  # Clear skill_ids for ObjectId references
        
        # for skill_id in skill_ids:
        #     # Get the skill data from the skill_id
        #     sk_id = db['skills'].insert_one({'skill': skill_id}).inserted_id
        #     stage['skill_ids'].append(sk_id)
        
        # Insert each stage entry into the 'stages' collection
        stage_id = db['stage'].insert_one(stage).inserted_id
        person['stages'].append(stage_id)  # Store the generated ObjectId
    
    # Insert the person with references to education and stages
    db['cpMapEntry'].insert_one(person)

print("Data inserted successfully!")

Data inserted successfully!
